## Prepare ADRD dataset

In [1]:
## Load packages ----
import numpy as np
import pandas as pd
import feather

import sshtunnel
import psycopg2 as pg

import os
import json
import sys

import seaborn as sns
import matplotlib.pyplot as plt

/n/home_fasse/maudirac/.conda/envs/medicare_QC/lib/python3.9/site-packages/paramiko/transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


## Zip to county crosswalk

In [2]:
## read zip to county crosswalk ----
zip_to_county_1 = pd.read_csv('../data/input/remote/zip_county_2010.csv')
zip_to_county_1 = zip_to_county_1[['ZIP', 'COUNTY']]
zip_to_county_1 = zip_to_county_1.rename(columns = {'ZIP':'zip', 'COUNTY':'county'})
zip_to_county_1

,zip,county
0,501,36103
1,544,36103
2,601,72001
3,602,72003
4,603,72005
...,...,...
46870,99925,2201
46871,99926,2201
46872,99927,2201
46873,99928,2130


In [3]:
zip_to_county_1.isna().sum()

zip       0
county    0
dtype: int64

In [4]:
## Open ssh tunnel to DB host ----
tunnel = sshtunnel.SSHTunnelForwarder(
    ('nsaph.rc.fas.harvard.edu', 22),
    ssh_username=f'{os.environ["MY_NSAPH_SSH_USERNAME"]}',
    ssh_private_key=f'{os.environ["HOME"]}/.ssh/id_rsa', 
    ssh_password=f'{os.environ["MY_NSAPH_SSH_PASSWORD"]}', 
    remote_bind_address=("localhost", 5432)
)

tunnel.start()

## Open connection to DB ----
connection = pg.connect(
    host='localhost',
    database='nsaph2',
    user=f'{os.environ["MY_NSAPH_DB_USERNAME"]}',
    password=f'{os.environ["MY_NSAPH_DB_PASSWORD"]}', 
    port=tunnel.local_bind_port
)

In [5]:
## Define query ----
sql_query = f"""
SELECT *
FROM public.hud_zip2fips  
WHERE year = '2015';
"""
## Request query ----
zip_to_county_2 = pd.read_sql_query(sql_query, connection, index_col = 'zip').reset_index()

In [6]:
zip_to_county_2 = zip_to_county_2[['zip', 'county']]
zip_to_county_2

,zip,county
0,501,36103
1,601,72001
2,602,72003
3,603,72005
4,603,72071
...,...,...
51677,99925,2198
51678,99926,2198
51679,99927,2198
51680,99928,2130


In [7]:
county1 = set(zip_to_county_1.county)
county2 = set(zip_to_county_2.county)

In [8]:
np.array(county1.difference(county2))

array({2280, 51560, 2232, 2201, 51515}, dtype=object)

In [9]:
np.array(county2.difference(county1))

array({2275, 8014, 2195, 2230, 2198, 46102, 2105, 66010}, dtype=object)

* use crosswalk from 2015

In [10]:
zip_to_county_1[zip_to_county_1.county == 8014]

,zip,county


In [11]:
zip_to_county_2[zip_to_county_2.county == 8014]

,zip,county
44317,80020,8014
44320,80021,8014
44324,80023,8014
44328,80027,8014
44339,80038,8014
44463,80234,8014
44607,80516,8014
44650,80603,8014


In [12]:
## get weights ----
zip_w = zip_to_county_2.groupby(['zip'])['county'].count().reset_index()
zip_w = zip_w.rename(columns = {'county':'w'})
zip_w['w'] = 1 / zip_w.w
zip_to_county = zip_to_county_2.merge(zip_w)
zip_to_county.w.describe()

count    51682.000000
mean         0.764405
std          0.282065
min          0.166667
25%          0.500000
50%          1.000000
75%          1.000000
max          1.000000
Name: w, dtype: float64

In [13]:
zip_to_county[zip_to_county.county == 8014]

,zip,county,w
44317,80020,8014,0.250000
44320,80021,8014,0.500000
44324,80023,8014,0.500000
44328,80027,8014,0.500000
44339,80038,8014,1.000000
44463,80234,8014,0.500000
44607,80516,8014,0.333333
44650,80603,8014,0.333333


In [14]:
adm_zip_df = pd.read_feather("../data/input/local/adm_zip_df.feather")
adm_zip_df.shape

(17429040, 6)

In [15]:
## crosswalk to counties ----
adm_county_df = adm_zip_df.merge(zip_to_county)
adm_county_df.isna().sum()

year       0
zip        0
race       0
sex        0
age_grp    0
n_adrd     0
county     0
w          0
dtype: int64

In [16]:
adm_county_df[['year', 'race']].groupby(['race']).count()

,year
race,
1,7186320
2,7186320
5,7186320


In [17]:
adm_county_df['n_adrd'] = adm_county_df.n_adrd * adm_county_df.w
adm_county_df = adm_county_df.groupby(['year', 'county', 'race', 'sex', 'age_grp'], 
                      as_index = False, 
                      observed = True, 
                      dropna = True)['n_adrd'].sum()
adm_county_df

,year,county,race,sex,age_grp,n_adrd
0,2000,1001,1,1,<65,0.0
1,2000,1001,1,1,"[65,75)",6.0
2,2000,1001,1,1,"[75,85)",15.5
3,2000,1001,1,1,>85,15.5
4,2000,1001,1,2,<65,0.0
...,...,...,...,...,...,...
1391035,2018,72153,5,1,>85,11.0
1391036,2018,72153,5,2,<65,0.0
1391037,2018,72153,5,2,"[65,75)",0.0
1391038,2018,72153,5,2,"[75,85)",6.5


In [18]:
adm_county_df[['year', 'age_grp']].groupby(['age_grp']).count()

,year
age_grp,
<65,347760
"[65,75)",347760
"[75,85)",347760
>85,347760


In [19]:
adm_zip_df.isna().sum()

year       0
zip        0
race       0
sex        0
age_grp    0
n_adrd     0
dtype: int64

In [20]:
## total number of adrd admissions in zipcodes ----
adm_zip_df.n_adrd.sum()

29076560

In [21]:
## total number of adrd admissions in counties ----
adm_county_df.n_adrd.sum()

28980848.999999985

In [22]:
#adm_county_df.to_csv("../data/input/local/adm_county_df.csv", index=False)
adm_county_df.to_feather("../data/input/local/adm_county_df.feather")

## Adrd counts

In [23]:
adm_county_df = pd.read_feather("../data/input/local/adm_county_df.feather")
bene_county_df = pd.read_feather("../data/input/local/bene_county_df.feather")

In [24]:
bene_county_df

,year,county,race,sex,age_grp,n_enrollees
0,2000,1001,1,1,<65,435.333333
1,2000,1001,1,1,>85,87.166667
2,2000,1001,1,1,"[65,75)",843.666667
3,2000,1001,1,1,"[75,85)",371.166667
4,2000,1001,1,2,<65,386.000000
...,...,...,...,...,...,...
833447,2018,72153,2,2,<65,8.000000
833448,2018,72153,2,2,>85,12.000000
833449,2018,72153,2,2,"[65,75)",32.000000
833450,2018,72153,2,2,"[75,85)",26.000000


In [25]:
## obtain rows for all combinations of county, year, race, sex and age_grp ----
## merge with enrollee and adrd counts
## there may be missing counts for a given combination
county_ = sorted(bene_county_df.county.unique())
year_ = sorted(bene_county_df.year.unique())
race_ = sorted(bene_county_df.race.unique())
sex_ = sorted(bene_county_df.sex.unique())
age_grp_ = sorted(bene_county_df.age_grp.unique())

adrd_county_df = pd.DataFrame({'county':county_}).merge(pd.DataFrame({'year':year_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'race':race_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'sex':sex_}), how = 'cross')
adrd_county_df = adrd_county_df.merge(pd.DataFrame({'age_grp':age_grp_}), how = 'cross')
adrd_county_df.shape

(927648, 5)

In [26]:
## add state ----
adrd_county_df['state'] = [int(float(x)/1000) for x in adrd_county_df.county]

In [27]:
adrd_county_df = adrd_county_df.merge(bene_county_df, how = 'left')
adrd_county_df = adrd_county_df.merge(adm_county_df, how = 'left')
adrd_county_df

,county,year,race,sex,age_grp,state,n_enrollees,n_adrd
0,1001,2000,1,1,<65,1,435.333333,0.0
1,1001,2000,1,1,>85,1,87.166667,15.5
2,1001,2000,1,1,"[65,75)",1,843.666667,6.0
3,1001,2000,1,1,"[75,85)",1,371.166667,15.5
4,1001,2000,1,2,<65,1,386.000000,0.0
...,...,...,...,...,...,...,...,...
927643,78030,2018,2,1,"[75,85)",78,NaN,NaN
927644,78030,2018,2,2,<65,78,NaN,NaN
927645,78030,2018,2,2,>85,78,NaN,NaN
927646,78030,2018,2,2,"[65,75)",78,NaN,NaN


In [28]:
## there are county-years-race-sex-age_grps with zero enrollees ----
adrd_county_df[adrd_county_df.n_enrollees.isnull()]

,county,year,race,sex,age_grp,state,n_enrollees,n_adrd
19113,1133,2006,2,1,>85,1,NaN,0.0
19129,1133,2007,2,1,>85,1,NaN,0.0
19145,1133,2008,2,1,>85,1,NaN,0.0
19161,1133,2009,2,1,>85,1,NaN,0.0
19177,1133,2010,2,1,>85,1,NaN,0.0
...,...,...,...,...,...,...,...,...
927643,78030,2018,2,1,"[75,85)",78,NaN,NaN
927644,78030,2018,2,2,<65,78,NaN,NaN
927645,78030,2018,2,2,>85,78,NaN,NaN
927646,78030,2018,2,2,"[65,75)",78,NaN,NaN


In [29]:
## total number of counties in bene_county_df (resulting from crosswalk)----
len(county_)

3221

In [30]:
## total number of counties in adrd_county_df ----
len(adrd_county_df.county.unique())

3221

In [31]:
## total number of enrollee-years in adrd_county_df ----
adrd_county_df.n_enrollees.sum()

792564236.0

In [32]:
## total number of adrd admissions in adrd_county_df ----
adrd_county_df.n_adrd.sum()

28358747.16666668

In [33]:
## percentage of county-race-sex-age_grp combinations with missing enrollees (all years) ----
adrd_county_df.n_enrollees.isnull().mean()

0.10154282658939598

In [34]:
## percentage of county-race-sex-age_grp combinations with missing adrd hospitalizations (all years) ----
adrd_county_df.n_adrd.isnull().mean()

0.0027941633033219497

In [35]:
(adrd_county_df.n_adrd == 0).mean()

0.3293760133153955

In [36]:
# strange cases where there is a hospitalization recorded where no enrollees are registered
adrd_county_df[adrd_county_df.n_enrollees.isnull() & 
               adrd_county_df.n_adrd.notnull() & 
               adrd_county_df.n_adrd != 0]

,county,year,race,sex,age_grp,state,n_enrollees,n_adrd
28395,4009,2010,2,1,"[75,85)",4,NaN,1.000000
28411,4009,2011,2,1,"[75,85)",4,NaN,1.000000
32141,5005,2010,2,2,>85,5,NaN,2.000000
32157,5005,2011,2,2,>85,5,NaN,2.000000
38171,5047,2009,2,1,"[75,85)",5,NaN,0.500000
...,...,...,...,...,...,...,...,...
900505,56017,2013,2,1,>85,56,NaN,0.333333
900521,56017,2014,2,1,>85,56,NaN,0.666667
904235,56043,2012,2,1,"[75,85)",56,NaN,0.333333
912414,72054,2001,2,2,"[65,75)",72,NaN,1.000000


In [37]:
## save adrd_county_df
adrd_county_df.to_csv("../data/input/local/adrd_county_df.csv", index=False)
#adrd_county_df.to_feather("../data/input/local/adrd_county_df.feather")